# 🌌 MURE AGI: The Great Distillation (5M Rules)
### Pipeline: Extraction -> Distillation -> Deployment
**Created by Myo Min Aung**

This system handles the autonomous generation of 5 million causal rules and distills the logic from Gemma-2-2B into our specialized Sentence-based LLM.

## 1. Environment & Keep-Alive
Mounting Drive and starting the anti-idle script.

In [ ]:
from google.colab import drive
import os
from IPython.display import display, Javascript

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/svo cc brain"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)

# Keep-Alive Script
display(Javascript('''
    function ClickConnect() {
        console.log("✅ Keep-alive: Clicking Connect button...");
        document.querySelector("colab-connect-button").click()
    }
    setInterval(ClickConnect, 60000)
'''))

print(f"✅ Workspace: {BASE_DIR}")

## 2. Dependencies
Installing specialized libraries for high-speed NLP and training.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes peft datasets sentencepiece aiohttp jsonlines huggingface_hub


## 3. Persistent 5M Rule Extractions
This cell checks if `rules_5m.json` exists. If not, it starts/resumes extraction from Wikipedia until 5,000,000 rules are collected.

In [ ]:
import json
import re
import requests
import time
import jsonlines
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

RULES_FILE = os.path.join(BASE_DIR, "rules_5m.jsonl")
TARGET_COUNT = 5_000_000

def load_existing_rules():
    rules = []
    if os.path.exists(RULES_FILE):
        try:
            with jsonlines.open(RULES_FILE) as reader:
                for obj in reader:
                    rules.append(obj)
        except Exception as e:
            print(f"Warning reading jsonl: {e}")
    return rules

def extract_causal_logic(text):
    patterns = [
        r"(.*) causes (.*)",
        r"(.*) leads to (.*)",
        r"(.*) results in (.*)",
        r"because (.*), (.*)"
    ]
    found = []
    sentences = re.split(r'[.!?]\s+', text)
    for s in sentences:
        for p in patterns:
            match = re.search(p, s.lower())
            if match:
                c, e = match.groups()
                if len(c.strip()) > 5 and len(e.strip()) > 5:
                    found.append({"cause": c.strip(), "effect": e.strip()})
    return found

rules = load_existing_rules()
seen_keys = set((r['cause'], r['effect']) for r in rules)
print(f"📊 Existing Rules: {len(rules):,}")

from datasets import load_dataset

if len(rules) < TARGET_COUNT:
    print("🚀 Extracting from HF Wikipedia dump (Blazing Fast, No API Limits!)...")
    pbar = tqdm(total=TARGET_COUNT, initial=len(rules))
    writer = jsonlines.open(RULES_FILE, mode='a')
    
    # Use streaming dataset to avoid downloading the entire dump at once
    dataset = load_dataset('wikipedia', '20220301.en', split='train', streaming=True)
    
    for record in dataset:
        text = record.get('text', '')
        new_links = extract_causal_logic(text)
        
        for link in new_links:
            key = (link['cause'], link['effect'])
            if key not in seen_keys:
                seen_keys.add(key)
                rules.append(link)
                writer.write(link)
                pbar.update(1)

        if len(rules) >= TARGET_COUNT:
            break
    
    writer.close()
    print("\n✅ 5,000,000 Rules Secured with HF Dataset Pipeline.")
else:
    print("✅ 5M Dataset already complete. Skipping extraction.")

## 4. Distillation Engine Setup
Loading the Teacher (Gemma) and Student models.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LlamaConfig, LlamaForCausalLM
from huggingface_hub import login
import torch

# Uncomment the below line and add your token if not using colab's secret manager
# login(token="YOUR_HF_TOKEN")

model_id = "google/gemma-2-2b-it"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

print("📥 Loading Teacher (Gemma)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
teacher = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

print("🏗️ Initializing Student (SentenceLLM-400M)...")
student_config = LlamaConfig(
    vocab_size=len(tokenizer), 
    hidden_size=1024, 
    intermediate_size=4096, 
    num_hidden_layers=12,
    num_attention_heads=16,
    num_key_value_heads=8,
    torch_dtype="bfloat16"
)
student = LlamaForCausalLM(student_config).to(torch.bfloat16).to("cuda")
student.gradient_checkpointing_enable()

## 5. Knowledge Distillation with Auto-Resume
The core trainer that mimics Gemma's logic using the 5M rules.

In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

class DistillationTrainer:
    def __init__(self, t, s, base_path):
        self.teacher = t
        self.student = s
        self.checkpoint_dir = os.path.join(base_path, "checkpoints")
        self.opt = AdamW(self.student.parameters(), lr=4e-5)
        self.step = 0
        self._load_resume()

    def _load_resume(self):
        path = os.path.join(self.checkpoint_dir, "latest_distill.pt")
        if os.path.exists(path):
            ckpt = torch.load(path, map_location='cpu')
            self.student.load_state_dict(ckpt['model'])
            self.student.to("cuda")
            self.opt.load_state_dict(ckpt['opt'])
            self.step = ckpt['step']
            print(f"✅ Resumed distillation from Step {self.step}")
        
        # Initialize scheduler
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=1000000, eta_min=1e-6)
        if os.path.exists(path) and 'scheduler' in ckpt:
            self.scheduler.load_state_dict(ckpt['scheduler'])

    def save(self):
        path = os.path.join(self.checkpoint_dir, "latest_distill.pt")
        ckpt = {
            'model': self.student.state_dict(), 
            'opt': self.opt.state_dict(), 
            'scheduler': self.scheduler.state_dict(),
            'step': self.step
        }
        torch.save(ckpt, path)
        if self.step % 5000 == 0:
            torch.save(self.student.state_dict(), os.path.join(self.checkpoint_dir, f"mure_3b_s{self.step}.pt"))

    def train(self, loader):
        self.teacher.eval()
        self.student.train()
        GRAD_ACCUM = 4
        pbar = tqdm(total=len(loader), initial=self.step % len(loader))
        
        for i, (input_ids, mask) in enumerate(loader):
            # Simple skip logic for resume (approximation with shuffle)
            if self.step > 0 and i < (self.step % len(loader)):
                if i % 100 == 0: pbar.update(100)
                continue
            
            ids, mask = input_ids.to("cuda"), mask.to("cuda")
            with torch.no_grad(): 
                t_logits = self.teacher(ids, attention_mask=mask).logits
            
            s_logits = self.student(ids, attention_mask=mask).logits
            
            # Reshape for multidimensional KL loss
            batch, seq, vocab = s_logits.shape
            s_log_prob = F.log_softmax(s_logits.reshape(-1, vocab) / 2.0, dim=-1)
            t_prob = F.softmax(t_logits.reshape(-1, vocab) / 2.0, dim=-1)
            
            loss = F.kl_div(s_log_prob, t_prob, reduction='batchmean') * 4.0
            loss = loss / GRAD_ACCUM
            loss.backward()
            torch.cuda.empty_cache()
            
            if (i + 1) % GRAD_ACCUM == 0:
                self.opt.step()
                self.scheduler.step()
                self.opt.zero_grad()
            
            self.step += 1
            if self.step % 10 == 0:
                loss_val = loss.item()*GRAD_ACCUM
                pbar.set_description(f"Loss: {loss_val:.4f}")
                with open(os.path.join(self.checkpoint_dir, 'training_log.txt'), 'a') as f:
                    f.write(f'Step {self.step}: {loss_val:.4f}\n')
            if self.step % 500 == 0: self.save()
            pbar.update(1)
        self.save() # Final save

class RuleDataset(Dataset):
    def __init__(self, path, tk):
        print("📖 Mapping 5M rules (Virtual Mapping via HuggingFace Datasets for Efficiency)...")
        try:
            from datasets import load_dataset
            self.data = load_dataset('json', data_files=path, split='train')
        except Exception as e:
            print(f"❌ Failed to load rules: {e}")
            self.data = []
        self.tk = tk
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        try:
            rule = self.data[i]
            t = f"Cause: {rule.get('cause', '?')} -> Effect: {rule.get('effect', '?')}"
            e = self.tk(t, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
            return e['input_ids'].squeeze(), e['attention_mask'].squeeze()
        except Exception:
            # Fallback for errors
            import torch
            return torch.zeros(128, dtype=torch.long), torch.zeros(128, dtype=torch.long)

loader = DataLoader(RuleDataset(RULES_FILE, tokenizer), batch_size=8, shuffle=True)
trainer = DistillationTrainer(teacher, student, BASE_DIR)
trainer.train(loader)

## 6. Final Deployment Export
Exports the weights and tokenizers for MURE production server.

In [ ]:
torch.save(student.state_dict(), os.path.join(BASE_DIR, "mure_final_3b_weights.pt"))
tokenizer.save_pretrained(os.path.join(BASE_DIR, "tokenizer"))
print("🏆 Distillation Complete. MURE Brain is ready for deployment.")